# Brain Baseline

This is the **rules-first Genex brain**.

## What this notebook proves
- the system can route parent concerns into the right domains
- it can ask a few milestone questions
- it can estimate a developmental **range** rather than hallucinating an exact age
- it can return suitable activities
- the core estimate is rule-based, while retrieval adds grounding and explanation

## How to use this notebook in the meeting
1. Run all cells.
2. Highlight:
   - selected domains
   - estimated range per domain
   - suggested activities
   - what is rules-based vs RAG-based

In [6]:
import json
import re
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

DOMAINS = ["speech_language", "motor", "social_communication", "adaptive_cognitive"]

DOMAIN_LABELS = {
    "speech_language": "Speech / Language",
    "motor": "Motor",
    "social_communication": "Social / Communication",
    "adaptive_cognitive": "Adaptive / Cognitive",
}

ANSWER_WEIGHTS = {
    "yes": 1.0,
    "sometimes": 0.6,
    "with_help": 0.45,
    "not_yet": 0.0,
    "no": 0.0,
    "not_sure": 0.25,
}

SAFE_NOTE = (
    "This is a non-diagnostic developmental support estimate intended to guide "
    "parent activities and follow-up questions. It is not a medical diagnosis "
    "or formal developmental assessment."
)

AGE_BANDS = [
    (12, "0-12m"),
    (24, "12-24m"),
    (36, "24-36m"),
    (48, "36-48m"),
    (60, "48-60m"),
    (72, "60-72m"),
]

@dataclass
class MilestoneQuestion:
    qid: str
    domain: str
    milestone_month: int
    text: str
    weight: float = 1.0

@dataclass
class Activity:
    activity_id: str
    domain: str
    range_label: str
    max_minutes: int
    title: str
    instructions: str
    adaptation: str
    rationale_doc_id: str

@dataclass
class RetrievalDoc:
    doc_id: str
    doc_type: str
    title: str
    text: str
    tags: List[str]

MILESTONE_BANK = [
    MilestoneQuestion("sp_24_1", "speech_language", 24, "Does your child use at least 10-20 meaningful words?"),
    MilestoneQuestion("sp_36_1", "speech_language", 36, "Can your child combine 2-3 words to ask or comment?"),
    MilestoneQuestion("sp_48_1", "speech_language", 48, "Can your child answer simple who/what/where questions?"),
    MilestoneQuestion("sp_60_1", "speech_language", 60, "Can your child tell a simple story or describe a recent event?"),

    MilestoneQuestion("mo_24_1", "motor", 24, "Does your child walk independently most of the time?"),
    MilestoneQuestion("mo_36_1", "motor", 36, "Can your child run and stop without falling most of the time?"),
    MilestoneQuestion("mo_48_1", "motor", 48, "Can your child jump with both feet off the ground?"),
    MilestoneQuestion("mo_60_1", "motor", 60, "Can your child balance briefly on one foot or hop with support?"),

    MilestoneQuestion("so_24_1", "social_communication", 24, "Does your child show or bring things to share interest with you?"),
    MilestoneQuestion("so_36_1", "social_communication", 36, "Does your child join simple back-and-forth play with an adult?"),
    MilestoneQuestion("so_48_1", "social_communication", 48, "Can your child take turns in simple play or routines?"),
    MilestoneQuestion("so_60_1", "social_communication", 60, "Can your child play with peers using simple rules or roles?"),

    MilestoneQuestion("ad_24_1", "adaptive_cognitive", 24, "Can your child imitate simple household routines or gestures?"),
    MilestoneQuestion("ad_36_1", "adaptive_cognitive", 36, "Can your child complete a simple 2-step routine with prompts?"),
    MilestoneQuestion("ad_48_1", "adaptive_cognitive", 48, "Can your child follow a short daily routine with limited help?"),
    MilestoneQuestion("ad_60_1", "adaptive_cognitive", 60, "Can your child complete a short task and stay with it for a few minutes?"),
]

CONDITION_DOMAIN_PRIORS = {
    "autism": ["social_communication", "speech_language", "adaptive_cognitive"],
    "adhd": ["adaptive_cognitive", "social_communication"],
    "down syndrome": ["speech_language", "motor", "adaptive_cognitive"],
    "cerebral palsy": ["motor", "adaptive_cognitive"],
    "global developmental delay": ["speech_language", "motor", "social_communication", "adaptive_cognitive"],
    "speech delay": ["speech_language"],
}

KEYWORD_DOMAIN_MAP = {
    "speech_language": ["speech", "language", "talk", "words", "sentence", "understand", "communication", "verbal"],
    "motor": ["walk", "run", "jump", "balance", "grasp", "hand", "climb", "coordination", "motor"],
    "social_communication": ["social", "play", "peer", "eye contact", "name", "pretend", "turn taking", "interaction", "shared", "friends"],
    "adaptive_cognitive": ["routine", "attention", "focus", "dress", "feeding", "puzzle", "learn", "cognitive", "adaptive", "independent", "task"],
}

ACTIVITY_BANK = [
    Activity("act_sp_24_choice", "speech_language", "12-24m", 5, "Two-Choice Communication",
             "Offer two favorite items, name each one, and pause for a point, look, sound, or word before giving it.",
             "Accept gestures, signs, or approximated sounds as communication.",
             "doc_activity_speech_choice"),
    Activity("act_sp_36_expand", "speech_language", "24-36m", 7, "Word Expansion Play",
             "Repeat your child's word and add one more word, such as 'car' to 'blue car'.",
             "Keep it playful and use favorite toys or snacks.",
             "doc_activity_speech_expand"),
    Activity("act_sp_48_wh", "speech_language", "36-48m", 8, "Picture What/Where Game",
             "Use 3-4 pictures and ask simple what/where questions, giving a model answer first if needed.",
             "Let the child point before speaking if speaking is hard.",
             "doc_activity_speech_wh"),
    Activity("act_sp_60_story", "speech_language", "48-60m", 10, "Mini Story Retell",
             "After a short daily event, prompt your child to tell you who, what, and where in 2-3 sentences.",
             "Use pictures or sentence starters if needed.",
             "doc_activity_speech_story"),

    Activity("act_mo_24_obstacle", "motor", "12-24m", 5, "Cushion Obstacle Path",
             "Make a safe path with pillows or towels for stepping over and around objects.",
             "Provide hand support as needed.",
             "doc_activity_motor_obstacle"),
    Activity("act_mo_36_blocks", "motor", "24-36m", 7, "Stack and Drop",
             "Build short block towers together, then squat, reach, and rebuild.",
             "Use larger blocks if grasp is difficult.",
             "doc_activity_motor_blocks"),
    Activity("act_mo_48_jump", "motor", "36-48m", 8, "Jump-and-Stop Game",
             "Practice jumping in place and freezing when you say stop.",
             "Count down first if transitions are hard.",
             "doc_activity_motor_jump"),
    Activity("act_mo_60_balance", "motor", "48-60m", 10, "Line Walk and Balance",
             "Walk on a taped line, pause, and balance for a few seconds while reaching for a toy.",
             "Hand support is fine; reduce support gradually.",
             "doc_activity_motor_balance"),

    Activity("act_so_24_share", "social_communication", "12-24m", 5, "Show Me Game",
             "Take turns showing exciting objects and waiting for shared look or reaction.",
             "Celebrate eye gaze, pointing, or turning toward you.",
             "doc_activity_social_share"),
    Activity("act_so_36_turns", "social_communication", "24-36m", 7, "Turn-Taking with Favorite Toy",
             "Use a toy the child loves and alternate turns every few seconds with clear 'my turn/your turn' language.",
             "Keep turns very short at first.",
             "doc_activity_social_turns"),
    Activity("act_so_48_pretend", "social_communication", "36-48m", 8, "Pretend Play Routine",
             "Feed a doll or animal, take turns, and model one pretend action at a time.",
             "Use real objects first, then symbolic play.",
             "doc_activity_social_pretend"),
    Activity("act_so_60_peer", "social_communication", "48-60m", 10, "Simple Rule Game",
             "Play a short game with one rule, like rolling a ball and waiting for turns.",
             "Preview the rule before starting.",
             "doc_activity_social_rules"),

    Activity("act_ad_24_sort", "adaptive_cognitive", "12-24m", 5, "Same-or-Different Sort",
             "Sort 2-3 familiar objects into simple groups like spoons vs blocks.",
             "Start with very distinct objects.",
             "doc_activity_adaptive_sort"),
    Activity("act_ad_36_routine", "adaptive_cognitive", "24-36m", 7, "Two-Step Helper Job",
             "Give a short routine such as 'get socks, put in basket' using gesture plus words.",
             "Use the same routine daily for easier learning.",
             "doc_activity_adaptive_routine"),
    Activity("act_ad_48_cleanup", "adaptive_cognitive", "36-48m", 8, "Picture Cleanup Plan",
             "Show a simple picture cue for 2-3 cleanup steps and let your child follow it with support.",
             "Use praise after each finished step.",
             "doc_activity_adaptive_cleanup"),
    Activity("act_ad_60_task", "adaptive_cognitive", "48-60m", 10, "Mini Task Board",
             "Use a tiny 3-step task board for snack setup, dressing, or toy cleanup.",
             "Keep the task under 10 minutes and visually clear.",
             "doc_activity_adaptive_task"),
]

RETRIEVAL_DOCS = [
    RetrievalDoc("doc_cond_autism", "condition_profile", "Autism profile summary",
                 "Autism can affect social communication, flexible routines, and language development in different ways across children. Parent-friendly support often focuses on communication, shared attention, play, and predictable routines.",
                 ["autism", "social", "speech", "routine", "play"]),
    RetrievalDoc("doc_cond_down", "condition_profile", "Down syndrome profile summary",
                 "Down syndrome may be associated with developmental differences across speech, motor, and adaptive skills. Helpful supports often include repetition, visual cues, and short practice embedded in daily routines.",
                 ["down syndrome", "speech", "motor", "adaptive", "routine"]),
    RetrievalDoc("doc_cond_adhd", "condition_profile", "ADHD profile summary",
                 "ADHD can affect attention, transitions, self-regulation, and task completion. Parent guidance often emphasizes short routines, visual structure, and quick reinforcement.",
                 ["adhd", "attention", "focus", "adaptive", "routine"]),
    RetrievalDoc("doc_cond_gdd", "condition_profile", "Global developmental delay profile summary",
                 "Global developmental delay can involve several developmental domains at once. A short structured interview can help identify which domains need the most immediate support and what to prioritize at home.",
                 ["global developmental delay", "speech", "motor", "social", "adaptive"]),
    RetrievalDoc("doc_method_milestones", "milestone_method", "Milestone interview method",
                 "The interview asks a few structured milestone questions in likely domains, then estimates a developmental range instead of claiming a precise developmental age. This keeps the output safer and easier to defend.",
                 ["milestones", "developmental range", "non-diagnostic", "questions"]),
    RetrievalDoc("doc_activity_speech_choice", "activity_rationale", "Communication choice-making rationale",
                 "Choice-making creates repeated opportunities for a child to communicate through pointing, looking, signs, sounds, or words during a familiar daily routine.",
                 ["speech", "communication", "choice", "request"]),
    RetrievalDoc("doc_activity_speech_expand", "activity_rationale", "Word expansion rationale",
                 "Language expansion supports early expressive language by giving the child a slightly richer version of what they already said.",
                 ["speech", "language", "expansion", "expressive"]),
    RetrievalDoc("doc_activity_speech_wh", "activity_rationale", "What/where question rationale",
                 "Simple picture-based WH questions support comprehension and early sentence use without needing long conversations.",
                 ["speech", "questions", "comprehension", "pictures"]),
    RetrievalDoc("doc_activity_speech_story", "activity_rationale", "Story retell rationale",
                 "Retelling a recent event supports sequencing, expressive language, and basic narrative skills.",
                 ["speech", "story", "expressive", "narrative"]),
]

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()

def band_from_month(month: int) -> str:
    for upper, label in AGE_BANDS:
        if month <= upper:
            return label
    return AGE_BANDS[-1][1]

def route_domains(concern_text: str, known_condition: Optional[str] = None) -> Dict[str, float]:
    text = normalize_text(concern_text)
    scores = {d: 0.0 for d in DOMAINS}
    for domain, keywords in KEYWORD_DOMAIN_MAP.items():
        for kw in keywords:
            if kw in text:
                scores[domain] += 1.0
    if known_condition:
        kc = known_condition.lower().strip()
        for key, domains in CONDITION_DOMAIN_PRIORS.items():
            if key in kc:
                for d in domains:
                    scores[d] += 1.5
    if max(scores.values()) == 0:
        scores["speech_language"] = 1.0
        scores["adaptive_cognitive"] = 0.8
    return scores

def pick_domains(domain_scores: Dict[str, float], max_domains: int = 2) -> List[str]:
    ranked = sorted(domain_scores.items(), key=lambda x: (-x[1], x[0]))
    selected = [domain for domain, score in ranked if score > 0][:max_domains]
    return selected or ["speech_language"]

def questions_for_domain(domain: str, child_age_months: int, max_questions: int = 3) -> List[MilestoneQuestion]:
    bank = [q for q in MILESTONE_BANK if q.domain == domain]
    bank = sorted(bank, key=lambda q: q.milestone_month)
    candidate_months = sorted({q.milestone_month for q in bank})
    target = min(candidate_months, key=lambda m: abs(m - child_age_months))
    chosen_months = []
    if target - 12 in candidate_months:
        chosen_months.append(target - 12)
    chosen_months.append(target)
    if target + 12 in candidate_months:
        chosen_months.append(target + 12)
    selected = [q for q in bank if q.milestone_month in chosen_months]
    selected = sorted(selected, key=lambda q: q.milestone_month)
    return selected[:max_questions]

def estimate_domain_range(questions: List[MilestoneQuestion], answers: Dict[str, str]) -> Dict[str, object]:
    if not questions:
        return {"estimated_range": None, "confidence": 0.0, "summary": "No milestone evidence collected.", "evidence": [], "missing_data": True}

    weighted = []
    evidence = []
    for q in sorted(questions, key=lambda x: x.milestone_month):
        answer = answers.get(q.qid, "not_sure")
        score = ANSWER_WEIGHTS.get(answer, 0.25) * q.weight
        weighted.append((q.milestone_month, score))
        evidence.append({
            "question_id": q.qid,
            "question": q.text,
            "milestone_month": q.milestone_month,
            "answer": answer,
            "score": round(score, 2),
        })

    passed = [m for m, s in weighted if s >= 0.67]
    partial = [m for m, s in weighted if 0.35 <= s < 0.67]
    if passed:
        anchor = max(passed)
    elif partial:
        anchor = max(12, min(partial))
    else:
        anchor = max(12, min(m for m, _ in weighted) - 12)

    higher_failures = [m for m, s in weighted if m > anchor and s < 0.35]
    if higher_failures:
        upper = min(higher_failures)
        lower = max(12, upper - 12)
    else:
        lower = anchor
        upper = min(anchor + 12, 72)

    known_answers = sum(1 for q in questions if answers.get(q.qid, "not_sure") != "not_sure")
    completeness = known_answers / len(questions)
    consistency = 1.0
    for i in range(1, len(weighted)):
        prev_m, prev_s = weighted[i - 1]
        curr_m, curr_s = weighted[i]
        if curr_m > prev_m and curr_s > prev_s + 0.45:
            consistency -= 0.15

    confidence = max(0.35, min(0.95, 0.45 + 0.35 * completeness + 0.20 * consistency))
    return {
        "estimated_range": f"{lower}-{upper}m",
        "confidence": round(confidence, 2),
        "summary": f"Estimated current functional range is around {lower}-{upper} months based on {known_answers}/{len(questions)} answered milestone prompts.",
        "evidence": evidence,
        "missing_data": known_answers == 0,
    }

def pick_activities(domain: str, estimated_range: Optional[str], time_available_min: int, max_items: int = 1) -> List[Dict[str, object]]:
    if not estimated_range:
        return []
    try:
        lower = int(str(estimated_range).split("-")[0])
    except Exception:
        lower = 24
    target_band = band_from_month(lower + 1)
    candidates = [
        a for a in ACTIVITY_BANK
        if a.domain == domain and a.range_label == target_band and a.max_minutes <= max(time_available_min, 5)
    ]
    if not candidates:
        candidates = [a for a in ACTIVITY_BANK if a.domain == domain]
    return [asdict(a) for a in candidates[:max_items]]

def retrieve_context(concern_text: str, known_condition: Optional[str], selected_domains: List[str], activities: List[Dict[str, object]], top_k: int = 4) -> List[Dict[str, object]]:
    text = normalize_text(concern_text)
    tokens = set(re.findall(r"[a-z]+", text))
    if known_condition:
        tokens.update(re.findall(r"[a-z]+", known_condition.lower()))
    tokens.update(" ".join(selected_domains).replace("_", " ").split())
    needed_doc_ids = {a["rationale_doc_id"] for a in activities if "rationale_doc_id" in a}
    scored = []
    for doc in RETRIEVAL_DOCS:
        score = 0
        score += len(tokens.intersection(set(doc.tags)))
        if doc.doc_id in needed_doc_ids:
            score += 3
        if known_condition and known_condition.lower() in doc.title.lower():
            score += 2
        if score > 0:
            scored.append((score, doc))
    ranked = [doc for _, doc in sorted(scored, key=lambda x: (-x[0], x[1].title))[:top_k]]
    return [asdict(doc) for doc in ranked]

def run_genex_brain(
    *,
    child_age_months: int,
    concern_text: str,
    known_condition: Optional[str] = None,
    time_available_min: int = 10,
    parent_answers: Optional[Dict[str, str]] = None,
    max_domains: int = 2,
    max_questions_per_domain: int = 3,
) -> Dict[str, object]:
    parent_answers = parent_answers or {}
    domain_scores = route_domains(concern_text, known_condition)
    selected_domains = pick_domains(domain_scores, max_domains=max_domains)

    questions_by_domain = {}
    estimates_by_domain = {}
    all_activities = []

    for domain in selected_domains:
        qs = questions_for_domain(domain, child_age_months, max_questions=max_questions_per_domain)
        questions_by_domain[domain] = [asdict(q) for q in qs]
        domain_answer_map = {q.qid: parent_answers.get(q.qid, "not_sure") for q in qs}
        estimate = estimate_domain_range(qs, domain_answer_map)
        estimates_by_domain[domain] = estimate
        acts = pick_activities(domain, estimate["estimated_range"], time_available_min, max_items=1)
        all_activities.extend(acts)

    retrieval_context = retrieve_context(concern_text, known_condition, selected_domains, all_activities, top_k=5)

    return {
        "input_summary": {
            "child_age_months": child_age_months,
            "known_condition": known_condition,
            "concern_text": concern_text,
            "time_available_min": time_available_min,
        },
        "selected_domains": [
            {"domain_key": d, "domain_label": DOMAIN_LABELS[d], "routing_score": round(domain_scores[d], 2)}
            for d in selected_domains
        ],
        "questions_by_domain": questions_by_domain,
        "developmental_estimate_by_domain": estimates_by_domain,
        "recommended_activities": all_activities,
        "retrieved_context": retrieval_context,
        "safety_note": SAFE_NOTE,
    }

def explain_architecture():
    return {
        "rule_based": [
            "Routing parent concerns into likely developmental domains",
            "Selecting milestone questions near the child's chronological age",
            "Scoring answers into a developmental range and confidence",
            "Matching activities to the estimated range and daily time available",
        ],
        "rag_used_for": [
            "Retrieving known-condition context when provided",
            "Retrieving rationale for suggested activities",
            "Adding grounded explanation text around milestone method and activity choices",
        ],
    }

def build_rag_summary(result: Dict[str, object]) -> str:
    domains = ", ".join([d["domain_label"] for d in result["selected_domains"]])
    activity_titles = [a["title"] for a in result["recommended_activities"]]
    doc_titles = [doc["title"] for doc in result["retrieved_context"][:2]]
    return (
        f"Focused domains: {domains}. Suggested activities: {', '.join(activity_titles)}. "
        f"Retrieved supporting context includes: {', '.join(doc_titles)}."
    )

def build_parent_friendly_summary(result: Dict[str, object]) -> str:
    pieces = []
    for domain_key, estimate in result["developmental_estimate_by_domain"].items():
        pieces.append(f"{DOMAIN_LABELS[domain_key]} looks around {estimate['estimated_range']} with confidence {estimate['confidence']}.")
    activity_titles = [a["title"] for a in result["recommended_activities"]]
    return (
        "Here is a simple home plan: "
        + " ".join(pieces)
        + " Start with: "
        + ", ".join(activity_titles)
        + ". Keep the activities short, playful, and repeatable."
    )

In [7]:
DEMO_CASES = [
    {
        "name": "Case 1 - autism with speech/social concerns",
        "payload": {
            "child_age_months": 60,
            "concern_text": "He has autism, uses a few words, struggles with turn taking, and does not play much with other kids.",
            "known_condition": "autism",
            "time_available_min": 10,
            "parent_answers": {"sp_48_1": "not_yet", "sp_60_1": "no", "so_48_1": "sometimes", "so_60_1": "no"},
        },
    },
    {
        "name": "Case 2 - down syndrome with motor/adaptive needs",
        "payload": {
            "child_age_months": 48,
            "concern_text": "She has down syndrome and needs help with movement, routines, and self-help skills.",
            "known_condition": "down syndrome",
            "time_available_min": 8,
            "parent_answers": {"mo_36_1": "yes", "mo_48_1": "with_help", "ad_36_1": "sometimes", "ad_48_1": "with_help"},
        },
    },
    {
        "name": "Case 3 - no diagnosis, parent reports language delay",
        "payload": {
            "child_age_months": 36,
            "concern_text": "My child is not talking much yet and gets frustrated when trying to ask for things.",
            "known_condition": None,
            "time_available_min": 5,
            "parent_answers": {"sp_24_1": "sometimes", "sp_36_1": "not_yet", "ad_24_1": "yes", "ad_36_1": "sometimes"},
        },
    },
]

In [8]:
result = run_genex_brain(**DEMO_CASES[0]["payload"])
result

{'input_summary': {'child_age_months': 60,
  'known_condition': 'autism',
  'concern_text': 'He has autism, uses a few words, struggles with turn taking, and does not play much with other kids.',
  'time_available_min': 10},
 'selected_domains': [{'domain_key': 'social_communication',
   'domain_label': 'Social / Communication',
   'routing_score': 3.5},
  {'domain_key': 'speech_language',
   'domain_label': 'Speech / Language',
   'routing_score': 2.5}],
 'questions_by_domain': {'social_communication': [{'qid': 'so_48_1',
    'domain': 'social_communication',
    'milestone_month': 48,
    'text': 'Can your child take turns in simple play or routines?',
    'weight': 1.0},
   {'qid': 'so_60_1',
    'domain': 'social_communication',
    'milestone_month': 60,
    'text': 'Can your child play with peers using simple rules or roles?',
    'weight': 1.0}],
  'speech_language': [{'qid': 'sp_48_1',
    'domain': 'speech_language',
    'milestone_month': 48,
    'text': 'Can your child answe

In [9]:
all_results = [{"case_name": case["name"], "result": run_genex_brain(**case["payload"])} for case in DEMO_CASES]
all_results

[{'case_name': 'Case 1 - autism with speech/social concerns',
  'result': {'input_summary': {'child_age_months': 60,
    'known_condition': 'autism',
    'concern_text': 'He has autism, uses a few words, struggles with turn taking, and does not play much with other kids.',
    'time_available_min': 10},
   'selected_domains': [{'domain_key': 'social_communication',
     'domain_label': 'Social / Communication',
     'routing_score': 3.5},
    {'domain_key': 'speech_language',
     'domain_label': 'Speech / Language',
     'routing_score': 2.5}],
   'questions_by_domain': {'social_communication': [{'qid': 'so_48_1',
      'domain': 'social_communication',
      'milestone_month': 48,
      'text': 'Can your child take turns in simple play or routines?',
      'weight': 1.0},
     {'qid': 'so_60_1',
      'domain': 'social_communication',
      'milestone_month': 60,
      'text': 'Can your child play with peers using simple rules or roles?',
      'weight': 1.0}],
    'speech_language':

In [10]:
explain_architecture()

{'rule_based': ['Routing parent concerns into likely developmental domains',
  "Selecting milestone questions near the child's chronological age",
  'Scoring answers into a developmental range and confidence',
  'Matching activities to the estimated range and daily time available'],
 'rag_used_for': ['Retrieving known-condition context when provided',
  'Retrieving rationale for suggested activities',
  'Adding grounded explanation text around milestone method and activity choices']}

## Talking points for the advisor

### Rule-based
- routing parent concerns into likely developmental domains
- selecting milestone questions near chronological age
- scoring answers into a developmental range and confidence
- matching activities to level and time available

### Retrieval / RAG
- condition context
- activity rationale
- grounded method explanation

### What this does **not** claim
- diagnosis
- exact developmental age
- clinical-grade assessment
- provider HIPAA workflow